In [7]:
USE {
	dependencies("org.duckdb:duckdb_jdbc:1.5.5.1")
}

In [2]:
%use dataframe
// %use lets-plot
// %use kandy
// %use plotly

import kotlinx.serialization.json.Json

In [3]:
import org.jetbrains.kotlinx.dataframe.io.DbConnectionConfig
import org.jetbrains.kotlinx.dataframe.schema.DataFrameSchema

In [4]:
val db = "/Users/darronporter/Downloads/Dissertation/code/output/results/gcloud_white_advantage_experiments_results_2026-08-31/white_advantage.duckdb"
val url = "jdbc:duckdb:$db"
val username = "duckdb"
val password = "password"

val dbConfig = DbConnectionConfig(url, username, password)

val tableName = "game_results"

In [5]:
val mm3 = "/Users/darronporter/Downloads/Dissertation/code/output/results/aws_white_advantage_2026-09-01/db.duckdb"
val url = "jdbc:duckdb:$mm3"
val username = "duckdb"
val password = "password"

val mmDbConfig = DbConnectionConfig(url, username, password)

In [13]:
val df = DataFrame.readSqlQuery(
	dbConfig,
	sqlQuery = """
WITH games AS (
    SELECT
        winner.winner.model      AS winner_model,
        winner.winner.depth      AS winner_depth,
        winner.winner.iterations AS winner_iterations,
        winner.winner.name       AS winner_colour,

        winner.loser.model       AS loser_model,
        winner.loser.depth       AS loser_depth,
        winner.loser.iterations  AS loser_iterations,
        winner.loser.name        AS loser_colour
    FROM game_results
),

normalized AS (
    SELECT
        CASE WHEN winner_model = 'MCTS' THEN winner_iterations ELSE loser_iterations END AS mcts_iter,
        CASE WHEN winner_model = 'MINIMAX' THEN winner_depth ELSE loser_depth END AS mm_depth,

        CASE WHEN winner_colour = 'WHITE' THEN winner_model ELSE loser_model END AS white_model,
        winner_model,
        winner_colour
    FROM games
)

SELECT
    mcts_iter AS "MCTS Iter",
    mm_depth  AS "MM Depth",
    white_model AS "Model",

    COUNT(*) FILTER (WHERE winner_colour = 'WHITE')    AS "WHITE Win",
    COUNT(*) FILTER (WHERE winner_colour = 'BLACK')    AS "BLACK Win",

    ROUND(
        COUNT(*) FILTER (WHERE winner_colour = 'WHITE') * 100.0 /
        COUNT(*), 4
    ) || '%' AS "WHITE Win %",

    ROUND(
        COUNT(*) FILTER (WHERE winner_colour = 'BLACK') * 100.0 /
        COUNT(*), 4
    ) || '%' AS "BLACK Win %",

   COUNT(*) as "Total Games"

FROM normalized
GROUP BY white_model, mcts_iter, mm_depth
ORDER BY mcts_iter ASC, mm_depth ASC
    """.trimIndent()
)

In [14]:
df

MCTS Iter,MM Depth,Model,WHITE Win,BLACK Win,WHITE Win %,BLACK Win %,Total Games
0,1,MCTS,504,496,50.4%,49.6%,1000
0,1,MINIMAX,421,579,42.1%,57.9%,1000
99,1,MCTS,488,512,48.8%,51.2%,1000
499,1,MCTS,477,523,47.7%,52.3%,1000
999,1,MCTS,517,483,51.7%,48.3%,1000


In [37]:
df.`WHITE Win`.sum().toDouble().times(100.0)/df.`Total Games`.sum().toDouble()

48.14

In [38]:
df.`BLACK Win`.sum().toDouble().times(100.0)/df.`Total Games`.sum().toDouble()

51.86

In [39]:
df.sum()

MCTS Iter,MM Depth,WHITE Win,BLACK Win,Total Games
1597,5,2407,2593,5000


In [27]:
val mm = DataFrame.readSqlQuery(
	mmDbConfig,
	sqlQuery = """
WITH games AS (
    SELECT
        winner.winner.model      AS winner_model,
        winner.winner.depth      AS winner_depth,
        winner.winner.iterations AS winner_iterations,
        winner.winner.name       AS winner_colour,

        winner.loser.model       AS loser_model,
        winner.loser.depth       AS loser_depth,
        winner.loser.iterations  AS loser_iterations,
        winner.loser.name        AS loser_colour
    FROM game_results
    WHERE winner_model IS NOT NULL
    LIMIT 1000
),

normalized AS (
    SELECT
        CASE WHEN winner_model = 'MCTS' THEN winner_iterations ELSE loser_iterations END AS mcts_iter,
        CASE WHEN winner_model = 'MINIMAX' THEN winner_depth ELSE loser_depth END AS mm_depth,

        CASE WHEN winner_colour = 'WHITE' THEN winner_model ELSE loser_model END AS white_model,
        winner_model,
        winner_colour
    FROM games
)

SELECT
    mcts_iter AS "MCTS Iter",
    mm_depth  AS "MM Depth",
    white_model AS "Model",

    COUNT(*) FILTER (WHERE winner_colour = 'WHITE')    AS "WHITE Win",
    COUNT(*) FILTER (WHERE winner_colour = 'BLACK')    AS "BLACK Win",

    ROUND(
        COUNT(*) FILTER (WHERE winner_colour = 'WHITE') * 100.0 /
        COUNT(*), 4
    ) || '%' AS "WHITE Win %",

    ROUND(
        COUNT(*) FILTER (WHERE winner_colour = 'BLACK') * 100.0 /
        COUNT(*), 4
    ) || '%' AS "BLACK Win %",

   COUNT(*) as "Total Games"

FROM normalized
GROUP BY white_model, mcts_iter, mm_depth
ORDER BY mcts_iter ASC, mm_depth ASC
    """.trimIndent()
)

In [28]:
mm

MCTS Iter,MM Depth,Model,WHITE Win,BLACK Win,WHITE Win %,BLACK Win %,Total Games
0,3,MINIMAX,448,552,44.8%,55.2%,1000


In [29]:
val combined = df.concat(mm)

In [30]:
combined

MCTS Iter,MM Depth,Model,WHITE Win,BLACK Win,WHITE Win %,BLACK Win %,Total Games
0,1,MCTS,504,496,50.4%,49.6%,1000
0,1,MINIMAX,421,579,42.1%,57.9%,1000
99,1,MCTS,488,512,48.8%,51.2%,1000
499,1,MCTS,477,523,47.7%,52.3%,1000
999,1,MCTS,517,483,51.7%,48.3%,1000
0,3,MINIMAX,448,552,44.8%,55.2%,1000


In [31]:
combined.sum()

MCTS Iter,MM Depth,WHITE Win,BLACK Win,Total Games
1597,8,2855,3145,6000


In [33]:
combined.`WHITE Win`.sum().toDouble().times(100.0)/combined.`Total Games`.sum().toDouble()

47.583333333333336

In [32]:
combined.`BLACK Win`.sum().toDouble().times(100.0)/combined.`Total Games`.sum().toDouble()

52.416666666666664